In [ ]:
!pip install wandb
!pip install transformers==4.28.0
!pip install torch 
!pip install -U scikit-learn
!pip install pandas 
!pip install numpy


  Using cached wandb-0.19.10-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (21.3 MB)
  Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Using cached docker_pycreds-0.4.0-py2.py3-none-any.whl (9.0 kB)
  Using cached GitPython-3.1.44-py3-none-any.whl (207 kB)
  Using cached protobuf-6.30.2-cp39-abi3-manylinux2014_x86_64.whl (316 kB)
  Using cached pydantic-2.11.3-py3-none-any.whl (443 kB)
  Using cached sentry_sdk-2.26.1-py2.py3-none-any.whl (340 kB)
  Using cached setproctitle-1.3.5-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (30 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.7/128.7 kB 54.0 MB/s eta 0:00:00
  Attempting uninstall: urll

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time
import os 
import wandb

In [ ]:
# Simple tweet dataset class
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Convert dict of tensors to tensors and remove batch dimension
        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
os.environ["WANDB_API_KEY"] = ""


def preprocess_tweet(text):
    """Preprocess tweet by replacing usernames and URLs with placeholders"""
    words = []
    for word in text.split():
        if word.startswith('@'):
            words.append('@user')
        elif word.startswith('http'):
            words.append('http')
        else:
            words.append(word)
    return ' '.join(words)

def analyze_sentiment(text, tokenizer, model, device='cpu'):
    """Analyze sentiment of a single text"""
    # Preprocess text
    text = preprocess_tweet(text)
    
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
    
    # sentiment_labels = ['Negative', 'Neutral', 'Positive']
    return prediction, probs[0][prediction].item()

def evaluate_model(df, tokenizer, model, device='cpu'):
    """Evaluate model on dataset and calculate average loss"""
    model = model.to(device)
    model.eval()
    correct = 0
    total = 0
    total_loss = 0  # Initialize total loss

    y_true = []
    y_pred = []

    # Create a DataLoader for the evaluation dataset
    eval_dataset = TweetDataset(
        df['tweet'].tolist(),
        df['sentiment'].tolist(),
        tokenizer
    )
    eval_dataloader = DataLoader(eval_dataset, batch_size=16, shuffle=False)

    # Use no_grad for evaluation
    with torch.no_grad():
        for batch in eval_dataloader:
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}

            # Forward pass
            outputs = model(**batch)
            logits = outputs.logits
            loss = outputs.loss  # Get loss for the batch
            total_loss += loss.item()  # Accumulate loss

            # Predictions
            probs = torch.softmax(logits, dim=1)
            predictions = torch.argmax(probs, dim=1)

            # Collect predictions and labels
            y_true.extend(batch['labels'].cpu().numpy())
            y_pred.extend(predictions.cpu().numpy())

            # Calculate accuracy
            correct += (predictions == batch['labels']).sum().item()
            total += batch['labels'].size(0)

    # Calculate metrics
    accuracy = correct / total
    avg_loss = total_loss / len(eval_dataloader)  # Average loss
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    # Print results
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Confusion Matrix:")
    print(conf_matrix)

    return accuracy,precision, recall, f1, avg_loss

def train_model(model, train_dataloader, val_df, tokenizer, device='cpu', epochs=10, lr=2e-5):
    """Train the model"""
    # Set up optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay = 0.01)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    
    # Training loop
    best_accuracy = 0
    model = model.to(device)
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 100)
        
        # Training
        model.train()
        total_loss = 0
        
        start_time = time.time()
        for i, batch in enumerate(train_dataloader):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Forward pass
            outputs = model(**batch)
            loss = outputs.loss
            total_loss += loss.item()
            
            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            # Print progress
            elapsed = time.time() - start_time
            if i % 100 == 0:
                print(f"Batch {i+1}/{len(train_dataloader)} | Loss: {loss.item():.4f} | Time: {elapsed:.2f}s")
                start_time = time.time()
        
        avg_train_loss = total_loss / len(train_dataloader)
        print(f"Average training loss: {avg_train_loss:.4f}")
        
        # Validation
        print("\nValidating...")
        val_accuracy, precision, recall, f1, avg_eval_loss = evaluate_model(val_df, tokenizer, model, device)

        wandb.log({
            'epoch': epoch + 1,
            'acc': val_accuracy,
            'precision': precision,
            'recall': recall, 
            'f1': f1,
            'eval_avg_loss': avg_eval_loss, 
            'train_loss': avg_train_loss
        })
        
        
        # Save best model
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            print(f"New best accuracy: {best_accuracy:.4f}")
            print("Saving model...")
            model.save_pretrained("./best_model_5")
            tokenizer.save_pretrained("./best_model_5")
    
    print(f"\nTraining completed! Best validation accuracy: {best_accuracy:.4f}")
    return model

def main():

    # wandb.init(project="tweet-sentiment-prediction")

    train_data_path = "sentiment_train_data.csv"  # Path to training data
    eval_data_path = "sentiment_valid_data.csv"  # Path to evaluation data
    test_data_path = "sentiment_test_data.csv"  # Path to test data

    model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"  # Model name
    mode = "eval"  # Mode: "train" or "eval"
    batch_size = 32  # Batch size for training
    epochs = 40  # Number of training epochs



    # Check for GPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    print("Loading datasets...")
    train_df = pd.read_csv(train_data_path)
    eval_df = pd.read_csv(eval_data_path)
    test_df = pd.read_csv(test_data_path)
    
    # Check if required columns exist
    for df, name in [(train_df, "train"), (eval_df, "validation"), (test_df, "test")]:
        if not all(col in df.columns for col in ['tweet', 'sentiment']):
            print(f"Error: {name} dataset must contain 'tweet' and 'sentiment' columns")
            return

    
    # Load tokenizer and model
    print(f"Loading model: {model_name}")
    try:
        if mode == "eval" and torch.cuda.is_available():
            # Try to load from best_model if it exists and we're in eval mode
            import os
            if os.path.exists("best_model_5"):
                print("Loading fine-tuned model from best_model")
                tokenizer = AutoTokenizer.from_pretrained("best_model_5")
                model = AutoModelForSequenceClassification.from_pretrained("best_model_5")
            else:
                tokenizer = AutoTokenizer.from_pretrained(model_name)
                model = AutoModelForSequenceClassification.from_pretrained(model_name)
        else:
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            config = AutoConfig.from_pretrained(model_name)
            config.hidden_dropout_prob = 0.1
            config.attention_probs_dropout_prob = 0.1
            model = AutoModelForSequenceClassification.from_pretrained(model_name, config = config)
           
    except Exception as e:
        print(f"Error loading model: {e}")
        print("Using older version of transformers might help if you see compiler errors")
        return
    
    if mode == "train":
        # Preprocess data
        print("Preprocessing tweets...")
        train_df['tweet'] = train_df['tweet'].apply(preprocess_tweet)
        eval_df['tweet'] = eval_df['tweet'].apply(preprocess_tweet)
        test_df['tweet'] = test_df['tweet'].apply(preprocess_tweet)

        # Create dataset and dataloader
        train_dataset = TweetDataset(
            train_df['tweet'].tolist(),
            train_df['sentiment'].tolist(),
            tokenizer
        )

        eval_dataset = TweetDataset(
            eval_df['tweet'].tolist(), 
            eval_df['sentiment'].tolist(),
            tokenizer
        )
        
        train_dataloader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
        eval_dataloader = DataLoader(eval_dataset,batch_size=batch_size,shuffle=False)
        
        # Train model
        print("Starting training...")
        model = train_model(
            model,
            train_dataloader,
            eval_df,
            tokenizer,
            device=device,
            epochs=epochs,
        )
        
    elif mode == "eval":
        # Evaluate model
        print("Evaluating model...")
        accuracy,precision, recall, f1, avg_loss = evaluate_model(test_df, tokenizer, model, device)
        
    

In [ ]:
main()